# Hinglish EDA

Load the raw TSV splits and compute class balance, lengths, vocabulary/OOV, and Zipf statistics.

In [2]:
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src import eda
import matplotlib.pyplot as plt

DATA_DIR = PROJECT_ROOT / 'data'
train = pd.read_csv(DATA_DIR / 'train.tsv', sep='\t', header=None, names=['uid', 'text_clean', 'label'])
val = pd.read_csv(DATA_DIR / 'validation.tsv', sep='\t', header=None, names=['uid', 'text_clean', 'label'])
test = pd.read_csv(DATA_DIR / 'test.tsv', sep='\t', header=None, names=['uid', 'text_clean'])
for frame in (train, val, test):
    frame['text_clean'] = frame['text_clean'].fillna('').astype(str)

class_counts = eda.compute_class_distribution(train)
lengths = eda.compute_length_stats(train, lambda text: text.split())
vocabulary = eda.compute_vocabulary_stats(train['text_clean'], test['text_clean'])
zipf = eda.compute_zipf_counts(train['text_clean'])
figures_dir = PROJECT_ROOT / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
class_counts.to_csv(figures_dir / 'class_distribution.csv', index=False)
lengths.to_csv(figures_dir / 'train_length_stats.csv', index=False)
pd.DataFrame([vocabulary]).to_csv(figures_dir / 'vocabulary_stats.csv', index=False)
zipf.to_csv(figures_dir / 'zipf_counts.csv', index=False)

ax = class_counts.plot.bar(x='label', y='count', legend=False, title='Class distribution')
ax.set_ylabel('Tweets')
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(figures_dir / 'class_distribution.png', dpi=150)
plt.close(fig)

ax = lengths['token_count'].plot.hist(bins=30, title='Training tweet token lengths')
ax.set_xlabel('Tokens')
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(figures_dir / 'token_length_histogram.png', dpi=150)
plt.close(fig)

class_counts, vocabulary, lengths.head(), zipf.head(10)


(   label  count
 0      0   4252
 1      1   5473
 2      2   4869,
 {'train_vocab_size': 30978,
  'test_vocab_size': 10914,
  'oov_rate': 0.30254718709913875},
    uid                                         text_clean  label  token_count  \
 0    3  pakistan ka ghra tauq pakistan israel ko tasle...      0           17   
 1   41  madarchod mull ye mathura nahi dikha tha jab m...      0           22   
 2   48  manya pradhan mantri mahoday shriman narendra ...      2           15   
 3   64            krishna jcb full trend chal rahi aa nan      2            8   
 4   66  loksabha janta sirf modi ko vote de rahi thi n...      2           16   
 
    character_count  
 0              103  
 1              102  
 2               99  
 3               39  
 4               73  ,
   token  count  rank
 0   nan   6399     1
 1   hai   4297     2
 2    ki   3836     3
 3    ko   3163     4
 4  nahi   2767     5
 5    ka   2450     6
 6    ke   2342     7
 7    se   1909     8
 8     k   16